# DM_G4_P0007_branch_and_bound_answer

## 0. 정답본 범위
- Gate: G4
- Phase: P0007
- Topic: 분지한계법, LP relaxation, bound, incumbent, pruning
- Problem notebook: DM_G4_P0007_branch_and_bound.ipynb
- Correct answers included: Yes
- Output language: Korean

## 1. 핵심 기준표
| 기준 | max 문제 기준 정답 |
|---|---|
| LP relaxation 값 `Z` | 현 노드 feasible region의 upper bound |
| incumbent `L` | 현재까지 발견한 가장 좋은 정수가능해 값 |
| branch | 소수값 `v`에 대해 `x <= floor(v)`와 `x >= ceil(v)` |
| bound pruning | `Z <= L`이면 더 좋은 정수해가 나올 수 없어 절단 |
| 최적성 증명 | 모든 open node가 절단되고 남은 incumbent가 있을 때 완료 |

## 2. 문항별 정답

### 문제 1 정답 — 배송로봇 정수 배치
**LP relaxation**
- 원래 정수조건 `x1, x2 integer`를 제거하고 `x1, x2 >= 0`인 연속 LP로 푼다.
- root `Z=260.0`은 max 정수문제에서 upper bound다.

**분기와 절단 과정**
| 노드 | 추가 제약 | LP 해 | Z | 판정 |
|---:|---|---|---:|---|
| 0 | 없음 | `(3.189, 4.324)` | 260.0 | fractional, `x1`로 branch |
| 1 | `x1 <= 3` | `(3.0, 4.4)` | 257.2 | fractional, `x2`로 branch |
| 2 | `x1 >= 4` | `(4.0, 2.5)` | 215.0 | 이후 `L=242`가 생기면 bound pruning |
| 3 | `x1 <= 3`, `x2 <= 4` | `(3.0, 4.0)` | 242.0 | integer, `L <- 242` |
| 4 | `x1 <= 3`, `x2 >= 5` | `(1.5, 5.0)` | 235.0 | `Z <= L`, 절단 |

**최적성**
- node 3에서 정수가능해 `(x1,x2)=(3,4)`와 `L=242`를 얻는다.
- node 2와 node 4의 upper bound가 모두 `L`보다 작거나 같으므로 더 좋은 정수해를 만들 수 없다.
- open node가 없으므로 최적 정수해는 `(3,4)`, 최적값은 `242`이다.

**오답 진단**
- root LP 해를 반올림하면 feasible 여부가 보장되지 않고, bound의 의미도 사라진다.
- branch-and-bound의 목적은 반올림이 아니라, 잘린 노드에서 더 좋은 정수해가 없음을 증명하는 것이다.


#### 문항별 시각화 학습자료 — 문제 1
- 시각화 목적: `Z` upper bound, `L` incumbent, branch/prune 사유를 한 트리에서 확인한다.
- 사용할 데이터: 배송로봇 문제의 LP relaxation 노드 0~4.
- 그래프 해석 포인트: 정수해 발견 전에는 `Z`가 커도 `L`이 갱신되지 않는다.
- 학생이 자주 하는 오해: fractional node의 `Z`를 incumbent로 착각한다.
- 체크포인트 질문: node 3 발견 뒤 node 2와 node 4를 왜 자를 수 있는가?


In [ ]:
# 시각화 업데이트: 문제 1 정답 B&B tree와 L/Z 업데이트
import matplotlib.pyplot as plt
import pandas as pd

rows = [
    {"node":0,"constraint":"root","x":"(3.189, 4.324)","Z":260.0,"L_after":"-inf","decision":"fractional -> branch x1","status":"fractional"},
    {"node":1,"constraint":"x1<=3","x":"(3.0, 4.4)","Z":257.2,"L_after":"-inf","decision":"fractional -> branch x2","status":"fractional"},
    {"node":3,"constraint":"x1<=3, x2<=4","x":"(3, 4)","Z":242.0,"L_after":"242","decision":"integer -> update L","status":"integer"},
    {"node":4,"constraint":"x1<=3, x2>=5","x":"(1.5, 5.0)","Z":235.0,"L_after":"242","decision":"Z<=L -> prune","status":"pruned"},
    {"node":2,"constraint":"x1>=4","x":"(4.0, 2.5)","Z":215.0,"L_after":"242","decision":"Z<=L -> prune","status":"pruned"},
]
df = pd.DataFrame(rows)
display(df)

pos = {0:(0.5,0.9),1:(0.25,0.6),2:(0.75,0.6),3:(0.12,0.3),4:(0.38,0.3)}
edges = [(0,1),(0,2),(1,3),(1,4)]
colors = {"fractional":"#fff1cc", "integer":"#d8efe4", "pruned":"#f4cccc"}
fig, ax = plt.subplots(figsize=(10,5))
for a,b in edges:
    ax.plot([pos[a][0],pos[b][0]],[pos[a][1],pos[b][1]], color="0.45")
for row in rows:
    nid = row["node"]
    x,y = pos[nid]
    label = f"Node {nid}\n{row['constraint']}\nx={row['x']}\nZ={row['Z']}\nL after={row['L_after']}\n{row['decision']}"
    ax.scatter(x,y,s=1600,color=colors[row["status"]],edgecolor="0.25",zorder=3)
    ax.text(x,y,label,ha="center",va="center",fontsize=8)
ax.set_title("Robot B&B: Z upper bound and L incumbent")
ax.axis("off")
plt.show()

print("관찰: node 3에서 처음 정수가능해가 발견되어 L=242가 된다.")
print("원인: max 문제에서 각 open node의 하위 정수해는 그 노드 Z를 넘을 수 없다.")
print("제한: 이 그림은 제공된 작은 2변수 예제의 tree만 보여준다.")
print("결론: node 2, node 4는 Z<=L이므로 최적해를 잃지 않고 절단된다.")


### 문제 2 정답 — 위성부품 0-1 배낭문제
**0-1 의미와 LP relaxation**
- `x_j=1`이면 부품 `j`를 탑재하고, `0`이면 탑재하지 않는다.
- relaxation: `0 <= x_j <= 1`로 바꾸고 같은 목적함수와 중량 제약을 유지한다.

**가치/무게 비율**
| 부품 | 가치/무게 |
|---:|---:|
| 2 | 3.7500 |
| 3 | 3.5714 |
| 1 | 3.5000 |
| 5 | 3.0000 |
| 6 | 3.0000 |
| 4 | 2.6667 |

**fractional bound**
- root relaxation은 비율 순서로 2, 3, 1, 5를 전부 넣고, 남은 1kg에 부품 6의 `1/5`만 넣으면 bound `118.0`이 된다.
- 이는 0-1 조건보다 넓은 feasible region에서 얻은 값이므로 0-1 정수문제의 upper bound다.

**첫 branch**
- `x1=1`: 남은 용량 22, fractional bound `118.0`.
- `x1=0`: 남은 용량 34, fractional bound `109.3333`.
- feasible integer solution `x=(1,1,1,0,1,0)`은 중량 33, 가치 `115`이므로 incumbent `L=115`가 된다.
- 이때 `x1=0` branch는 upper bound가 `115`보다 작으므로 prune할 수 있다.
- 제공 데이터 기준 최적해는 `x=(1,1,1,0,1,0)`, 최적값 `115`이다.

**오답 진단**
- 비율 순 greedy는 fractional knapsack에서는 bound 계산 방법이지만, 0-1 knapsack의 최적성 증명은 아니다.
- fractional solution의 `x6=0.2` 같은 값은 “부품의 20% 탑재”라는 비현실적 결정을 뜻하므로 0-1 해로 사용할 수 없다.
- Solver에서는 `x1:x6`에 bin option을 지정해야 한다.


#### 문항별 시각화 학습자료 — 문제 2
- 시각화 목적: fractional bound, include/exclude branch, incumbent pruning을 연결한다.
- 사용할 데이터: 위성부품 가치/무게와 용량 34kg.
- 그래프 해석 포인트: fractional bound는 넓어진 문제의 상한이며, 최종 0-1 선택 조합과 다르다.
- 학생이 자주 하는 오해: 가치/무게 비율 greedy를 그대로 0-1 최적해라고 단정한다.
- 체크포인트 질문: `x1=0` branch는 왜 incumbent가 생긴 뒤 절단되는가?


In [ ]:
# 시각화 업데이트: 문제 2 knapsack bound, incumbent, branch pruning
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

values = np.array([42,30,25,24,18,15])
weights = np.array([12,8,7,9,6,5])
items = np.arange(1,7)
ratio = values / weights
order = np.argsort(-ratio)
incumbent_x = np.array([1,1,1,0,1,0])
incumbent_value = int(values @ incumbent_x)
incumbent_weight = int(weights @ incumbent_x)

bound_rows = pd.DataFrame([
    {"branch":"root", "fixed":"없음", "fractional_bound":118.0, "integer_candidate":"-", "prune":"아직 아님"},
    {"branch":"x1=1", "fixed":"부품1 포함", "fractional_bound":118.0, "integer_candidate":"x=(1,1,1,0,1,0), value=115", "prune":"incumbent L=115"},
    {"branch":"x1=0", "fixed":"부품1 제외", "fractional_bound":109.3333, "integer_candidate":"상한이 L보다 작음", "prune":"Z<=L -> prune"},
])
display(bound_rows)

fig, axes = plt.subplots(1, 3, figsize=(14,4))
axes[0].bar(items, ratio, color=["#4c78a8" if i in order[:3] else "#bab0ac" for i in range(6)])
axes[0].set_xticks(items)
axes[0].set_title("value/weight ratio: bound order")
axes[0].set_xlabel("item")
axes[0].set_ylabel("value/weight")
for i,r in zip(items,ratio): axes[0].text(i,r+0.04,f"{r:.2f}",ha="center",fontsize=8)

axes[1].barh([0],[34],color="#e6e6e6",edgecolor="0.4")
left=0
for idx in order:
    take = min(weights[idx], max(0,34-left))
    if take <= 0: break
    axes[1].barh([0],[take],left=[left],color="#86bc86",edgecolor="white")
    label = f"j{idx+1}" if take == weights[idx] else f"j{idx+1}\nfrac {take:g}/{weights[idx]}"
    axes[1].text(left+take/2,0,label,ha="center",va="center",fontsize=8)
    left += take
axes[1].set_yticks([])
axes[1].set_xlim(0,36)
axes[1].set_title("root fractional bound = 118.0")
axes[1].set_xlabel("capacity used")

pos = {"root":(0.5,0.85), "x1=1":(0.25,0.45), "x1=0":(0.75,0.45)}
for a,b in [("root","x1=1"),("root","x1=0")]:
    axes[2].plot([pos[a][0],pos[b][0]],[pos[a][1],pos[b][1]],color="0.45")
labels = {
    "root":"root\nUB=118.0",
    "x1=1":f"x1=1\nUB=118.0\nL={incumbent_value}\nx={tuple(incumbent_x)}",
    "x1=0":"x1=0\nUB=109.33\nUB<L -> prune",
}
colors = {"root":"#fff1cc","x1=1":"#d8efe4","x1=0":"#f4cccc"}
for k,(x,y) in pos.items():
    axes[2].scatter(x,y,s=1500,color=colors[k],edgecolor="0.25")
    axes[2].text(x,y,labels[k],ha="center",va="center",fontsize=8)
axes[2].set_title("include/exclude branch and incumbent")
axes[2].axis("off")
plt.tight_layout()
plt.show()

print(f"관찰: incumbent x={tuple(int(v) for v in incumbent_x)}는 weight={incumbent_weight}, value={incumbent_value}이다.")
print("원인: x1=0 branch의 fractional upper bound가 109.33으로 L=115보다 낮다.")
print("제한: fractional bound는 조각 선택을 허용하므로 실제 0-1 해가 아니다.")
print("결론: 비율 greedy는 bound 계산 도구이고, 최적성은 incumbent와 branch pruning으로 증명한다.")


### 문제 3 정답 — 주어진 tree 해석
- node 0의 `Z=96.2`는 root relaxation upper bound다.
- `x1=3.4` 기준 branch는 `x1 <= 3`, `x1 >= 4`이다.
- node 3은 정수해이고 `Z=88.0`이므로 처음 발견 시 `L=88.0`으로 갱신한다.
- node 4와 node 7은 LP relaxation도 infeasible이므로 infeasible pruning이다.
- node 5는 정수해 `82.0`이지만 incumbent `88.0`보다 작으므로 최종해가 아닐 수 있다.
- node 6은 fractional이고 `Z=91.5 > L=88.0`이므로 더 좋은 정수해 가능성을 배제할 수 없어 branch한다.
- node 8은 정수해 `90.0`이고 기존 `L=88.0`보다 크므로 `L=90.0`으로 갱신한다.
- 남은 open node가 없고 모두 infeasible, bound 열세, integer 절단으로 종료되면 최적값은 `90.0`, 해는 `x1=4, x2=2`이다.

**학생 답안 진단**
- “node 0 해를 반올림하면 최적”은 틀렸다. LP relaxation 해는 upper bound일 뿐이고 반올림 feasible/optimal 보장이 없다.
- “정수해 하나 발견 후 바로 최적”도 틀렸다. 다른 open node의 upper bound가 incumbent보다 크면 더 좋은 정수해가 남아 있을 수 있다.

**source anchor / node_id**
- anchors: `DM_PDF03:p001:L001`, `DM_PDF03:p002:L001`, `DM_PDF03:p003:L001`, `DM_PDF03:p008:L001`, `DM_PDF03:p009:L001`, `DM_PDF04:p005:L002`, `DM_PDF04:p005:L006`, `DM_PDF04:p008:L002`, `DM_PDF04:p009:L002`, `DM_PDF04:p009:L006`, `DM_PDF04:p010:L002`
- node_id: `n_DM_PDF03.branch_and_bound`, `n_DM_PDF03.lp_relaxation`, `n_DM_PDF03.upper_lower_bound`, `n_DM_PDF03.branching_floor_ceil`, `n_DM_PDF03.pruning_rules`, `n_DM_PDF03.incumbent_solution`, `n_DM_PDF04.knapsack_problem`, `n_DM_PDF04.binary_variable`


#### 문항별 시각화 학습자료 — 문제 3
- 시각화 목적: 주어진 tree에서 `L` 갱신, infeasible pruning, bound pruning을 구분한다.
- 사용할 데이터: 문제 표의 node 0~8.
- 그래프 해석 포인트: node 8의 정수해가 node 3보다 좋아서 incumbent가 갱신된다.
- 학생이 자주 하는 오해: 정수해 하나가 나오면 즉시 전체 최적이라고 생각한다.
- 체크포인트 질문: node 6을 계속 branch해야 하는 이유를 `Z`와 `L`로 설명하라.


In [ ]:
# 시각화 업데이트: 문제 3 주어진 tree의 L/Z 갱신 완성판
import pandas as pd
import matplotlib.pyplot as plt

rows = [
    {"node":0,"Z":96.2,"state":"fractional","L_after":"-inf","reason":"branch x1<=3 / x1>=4"},
    {"node":1,"Z":94.6,"state":"fractional","L_after":"-inf","reason":"branch x2<=2 / x2>=3"},
    {"node":2,"Z":92.7,"state":"fractional","L_after":"-inf","reason":"branch x2<=1 / x2>=2"},
    {"node":3,"Z":88.0,"state":"integer","L_after":"88","reason":"첫 incumbent"},
    {"node":4,"Z":None,"state":"infeasible","L_after":"88","reason":"비가해 절단"},
    {"node":5,"Z":82.0,"state":"integer","L_after":"88","reason":"L보다 작아 갱신 없음"},
    {"node":6,"Z":91.5,"state":"fractional","L_after":"88","reason":"Z>L 이므로 계속 branch"},
    {"node":7,"Z":None,"state":"infeasible","L_after":"88","reason":"비가해 절단"},
    {"node":8,"Z":90.0,"state":"integer","L_after":"90","reason":"L 갱신, 최종 incumbent"},
]
df = pd.DataFrame(rows)
display(df)

pos = {0:(0.50,0.92),1:(0.25,0.70),2:(0.75,0.70),3:(0.12,0.48),4:(0.38,0.48),5:(0.62,0.48),6:(0.88,0.48),7:(0.76,0.26),8:(0.98,0.26)}
edges = [(0,1),(0,2),(1,3),(1,4),(2,5),(2,6),(6,7),(6,8)]
colors = {"fractional":"#fff1cc", "integer":"#d8efe4", "infeasible":"#f4cccc"}
fig, ax = plt.subplots(figsize=(12,5.8))
for a,b in edges:
    ax.plot([pos[a][0],pos[b][0]],[pos[a][1],pos[b][1]], color="0.5")
for row in rows:
    nid = row["node"]
    x,y = pos[nid]
    ztxt = "infeasible" if row["Z"] is None else f"Z={row['Z']}"
    label = f"Node {nid}\n{ztxt}\n{row['state']}\nL after={row['L_after']}"
    ax.scatter(x,y,s=1450,color=colors[row["state"]],edgecolor="0.25",zorder=3)
    ax.text(x,y,label,ha="center",va="center",fontsize=8)
ax.set_title("Given B&B tree: L update path and pruning")
ax.axis("off")
plt.show()

print("관찰: L은 node 3에서 88, node 8에서 90으로 갱신된다.")
print("원인: incumbent는 정수가능해일 때만 갱신되고 fractional node는 후보가 아니다.")
print("제한: node 선택 순서는 문제 표의 일부 탐색 순서를 따른다.")
print("결론: 모든 open node가 절단되면 L=90, x=(4,2)가 최적성 증명을 가진다.")


## 3. 채점 기준
| 항목 | 배점 | 부분점 기준 |
|---|---:|---|
| LP relaxation과 bound 방향 해석 | 25 | max 문제에서 `Z`가 upper bound임을 반대로 쓰면 큰 감점 |
| floor/ceil 또는 include/exclude branching | 20 | `round(v)` 하나로 branch하면 오답 |
| pruning 사유 구분 | 25 | infeasible, `Z <= L`, integer 절단 구분 |
| incumbent와 최적성 증명 | 20 | open node 종료 조건까지 설명 |
| source anchor/node_id/Solver 연결 | 10 | source와 bin/int 옵션 연결 |

## 4. 오답튜터 기준표
| 오류 | 진단 질문 | 교정 힌트 |
|---|---|---|
| LP relaxation 해를 반올림 | “그 반올림해가 모든 제약을 만족하는가?” | bound와 feasible integer solution을 분리한다. |
| max 문제 bound 방향 반대 | “LP feasible region이 원문제보다 넓은가?” | max에서는 LP relaxation 값이 upper bound다. |
| 정수해 하나로 바로 종료 | “다른 open node의 upper bound가 더 큰가?” | open node가 모두 절단되어야 한다. |
| `Z <= L` pruning 설명 실패 | “이 노드 아래 목적값이 상위 bound를 넘을 수 있는가?” | 하위 노드의 bound는 현재 `Z` 이하이다. |
| infeasible과 bound pruning 혼동 | “LP 자체가 불가능한가, 가능하지만 값이 낮은가?” | 사유를 표로 나누어 적는다. |
| floor/ceil branch 대신 round | “소수값이 들어갈 수 없는 두 영역을 모두 덮었는가?” | `<= floor(v)`와 `>= ceil(v)`를 함께 쓴다. |
| fractional solution으로 incumbent 갱신 | “incumbent가 정수가능해인가?” | incumbent는 정수조건을 만족해야 한다. |
| node selection과 branching variable selection 혼동 | “다음에 풀 노드 선택인가, 나눌 변수 선택인가?” | 두 선택을 별도로 설명한다. |

## 5. 다음 Phase 연결
- P0008에서는 0-1 변수들이 coverage matrix의 열 선택으로 확장된다.
- `Ax >= 1`, `Ax = 1`, `Ax <= 1`의 차이를 이해해 set covering, partitioning, packing을 구분한다.
- facility location도 내부적으로 branch-and-bound가 필요한 0-1 정수계획이다.
